# RAG Demo — Indexado del KB + Búsqueda híbrida (Fase 2.2)

**Proyecto Final — Asistente de Triaje de Mails con RAG y LLM**

---

## Objetivo

1. **Indexar** las 30 FAQs generadas en `data/kb/` con doble enfoque:
   - **Chroma** con embeddings `multilingual-e5-large` (búsqueda semántica).
   - **BM25** en memoria (búsqueda léxica).
2. **Probar el retriever híbrido** con Reciprocal Rank Fusion (RRF).
3. **Demostrar la capacidad multilingüe**: consultas en español recuperan FAQs en inglés correctamente.

## Lo que NO hace este notebook

Todavía **no hay LLM en el flujo**. Solo retrieval. La generación de respuestas viene en la Fase 3.

## Lo que sí prueba

- El KB se indexa correctamente.
- BM25 + vector + RRF funcionan integrados.
- Consultas multilingües recuperan documentos relevantes.
- Tenemos métricas base para Recall@k cuando construyamos el ground truth (Fase 4).

## 1. Setup

In [ ]:
%%capture
!pip install -q -U google-genai pydantic python-dotenv pandas
!pip install -q sentence-transformers chromadb rank-bm25
!pip uninstall -y google-generativeai 2>/dev/null || true

In [ ]:
REPO_URL = "https://github.com/<TU-USUARIO>/<TU-REPO>.git"  # ⚠️ cambialo por tu URL

import os, sys, json, time
from pathlib import Path

IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    repo_name = REPO_URL.rstrip('/').split('/')[-1].replace('.git', '')
    if not Path(repo_name).exists():
        os.system(f'git clone {REPO_URL}')
    os.chdir(repo_name)
    PROJECT_ROOT = Path.cwd()
    try:
        from google.colab import userdata
        os.environ['GOOGLE_API_KEY'] = userdata.get('GOOGLE_API_KEY')
    except Exception:
        pass  # No es estrictamente necesaria para retrieval-only
else:
    PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
    from dotenv import load_dotenv
    load_dotenv(PROJECT_ROOT / '.env')

sys.path.insert(0, str(PROJECT_ROOT))
import pandas as pd
print(f'✅ Entorno: {"Colab" if IN_COLAB else "Local"}')
print(f'✅ Project root: {PROJECT_ROOT}')

## 2. Verificar que el KB existe

Si esta celda falla, primero correr `03_kb_generation.ipynb`.

In [ ]:
kb_files = sorted((PROJECT_ROOT / 'data/kb').glob('*.md'))
assert len(kb_files) >= 10, f'Solo {len(kb_files)} FAQs encontradas. Corré 03_kb_generation primero.'
print(f'✅ {len(kb_files)} FAQs en data/kb/')
print('Ejemplos:', [f.name for f in kb_files[:5]])

## 3. Indexar el KB en Chroma + BM25

Esto descarga el encoder `multilingual-e5-large` (~2.2 GB) la primera vez. Después de eso, indexar las 30 FAQs tarda menos de 1 minuto.

In [ ]:
from src.kb_ingest import ingest_kb

n_chunks = ingest_kb(
    kb_dir='data/kb',
    chroma_persist_dir='data/chroma',
    bm25_path='data/bm25.pkl',
    embedding_model='intfloat/multilingual-e5-large',
    verbose=True,
)
print(f'\n📊 Total chunks indexados: {n_chunks}')

## 4. Smoke test del retriever — consulta en inglés

In [ ]:
from src.retriever import search, reset_caches
reset_caches()  # asegura recargar los índices recién creados

query_en = "I forgot my password and the recovery email isn't arriving, please help."
results = search(query_en, top_k=3)

print(f'Query: {query_en}\n')
for i, r in enumerate(results, 1):
    print(f'#{i} {r.category} / {r.topic}')
    print(f'    score={r.score:.4f}  bm25_rank={r.bm25_rank}  vector_rank={r.vector_rank}')
    print(f'    ' + r.text[:180].replace('\n', ' ') + '...')
    print()

## 5. Test multilingüe — consulta en español sobre KB en inglés

**El punto clave del enfoque multilingüe del proyecto.** El KB está en inglés, la consulta es en español, y queremos que igual recupere las FAQs relevantes.

In [ ]:
query_es = "Hola, no puedo iniciar sesión y olvidé mi contraseña, ¿qué hago?"
results = search(query_es, top_k=3)

print(f'Query (ES): {query_es}\n')
for i, r in enumerate(results, 1):
    print(f'#{i} {r.category} / {r.topic}')
    print(f'    score={r.score:.4f}  bm25_rank={r.bm25_rank}  vector_rank={r.vector_rank}')
    print(f'    ' + r.text[:180].replace('\n', ' ') + '...')
    print()

**Observación importante:** los rankings de BM25 y vector divergen. BM25 solo encuentra matches en queries en inglés (no "contraseña"); el ranking vectorial sí captura la similitud semántica multilingüe. **RRF combina lo mejor de los dos.**

## 6. Batería de consultas — cobertura de las 5 categorías

Probamos 10 consultas (5 categorías × 2 idiomas) y verificamos visualmente que el top-1 sea de la categoría correcta. Esto anticipa la métrica **Precision@1** que vamos a medir con ground truth en la Fase 4.

In [ ]:
QUERIES = [
    ('en', 'ACCOUNT', "I want to enable two-factor authentication on my account."),
    ('es', 'ACCOUNT', "¿Cómo elimino mi cuenta?"),
    ('en', 'ORDER',   "How can I cancel an order I placed yesterday?"),
    ('es', 'ORDER',   "Mi pedido no llegó, ¿qué hago?"),
    ('en', 'REFUND',  "My card was charged twice for the same purchase."),
    ('es', 'REFUND',  "¿Cuánto tarda en llegarme un reembolso?"),
    ('en', 'PAYMENT', "Can I update the credit card I have on file?"),
    ('es', 'PAYMENT', "¿Cómo descargo mi factura?"),
    ('en', 'CONTACT', "What are your customer service hours?"),
    ('es', 'CONTACT', "Necesito hablar con un humano por favor."),
]

rows = []
for lang, expected_cat, q in QUERIES:
    results = search(q, top_k=3)
    top = results[0]
    rows.append({
        'lang':        lang,
        'query':       q[:60] + ('...' if len(q) > 60 else ''),
        'expected':    expected_cat,
        'top1_cat':    top.category,
        'ok':          '✅' if top.category == expected_cat else '❌',
        'top1_topic':  top.topic,
        'score':       round(top.score, 4),
    })
df = pd.DataFrame(rows)
df

In [ ]:
precision_at_1 = (df['ok'] == '✅').mean()
print(f'\nPrecision@1 (categoría correcta): {precision_at_1:.1%}')
print(f'  - EN: {(df[df["lang"] == "en"]["ok"] == "✅").mean():.1%}')
print(f'  - ES: {(df[df["lang"] == "es"]["ok"] == "✅").mean():.1%}')

## 7. Visualización del comportamiento de BM25 vs Vector

Para entender qué aporta cada uno, mostramos para una consulta cuántos documentos del top-K vinieron de cada método.

In [ ]:
def analyze_methods(query: str, top_k: int = 5):
    results = search(query, top_k=top_k)
    only_bm25  = sum(1 for r in results if r.bm25_rank is not None and r.vector_rank is None)
    only_vec   = sum(1 for r in results if r.vector_rank is not None and r.bm25_rank is None)
    both       = sum(1 for r in results if r.bm25_rank is not None and r.vector_rank is not None)
    print(f'Query: {query[:70]}')
    print(f'  Solo BM25 (léxico):   {only_bm25} de {top_k}')
    print(f'  Solo Vector (semánt): {only_vec} de {top_k}')
    print(f'  En ambos:             {both} de {top_k}')
    print()

analyze_methods("I forgot my password and the recovery email isn't arriving.")
analyze_methods("Hola, no puedo iniciar sesión y olvidé mi contraseña.")
analyze_methods("My card was charged twice.")

## 8. Conclusión de la Fase 2

- ✅ KB sintético de 30 FAQs generado y versionado.
- ✅ Indexado dual funcionando: Chroma (semántico) + BM25 (léxico).
- ✅ Retriever híbrido con RRF retornando resultados coherentes.
- ✅ Capacidad multilingüe demostrada: consultas en ES recuperan FAQs en EN gracias al encoder `multilingual-e5-large`.
- ✅ Precision@1 sobre la batería rápida queda anotada para la sección de Evaluación del informe.

## Próximo paso — Fase 3

Cerrar el círculo del RAG: tomar la salida del retriever, inyectarla como contexto en el prompt de Gemini, y que la respuesta del LLM cite las fuentes. Esto activa el campo `suggested_response` del `MailAnalysis` y nos da el sistema completo.

Después, en Fase 4, construimos el ground truth (consulta → FAQ esperada) y medimos las métricas oficiales del informe: Recall@k, Precision@k, nDCG, MRR, Faithfulness, Answer Relevance.